# Лабораторная работа №1
## Первичное исследование и оценка качества данных

**Датасет:** Hotel Booking Demand (`hotel_bookings.csv`)

Цель работы — выполнить первичное исследование датасета, провести аудит качества данных и выполнить мини-EDA.

## 1. Постановка задачи

### Что это за данные
Датасет содержит сведения о бронированиях двух типов отелей: `City Hotel` и `Resort Hotel`. Для каждого бронирования указаны сроки и параметры проживания, сведения о гостях, стране, способе оформления бронирования, типе номера, стоимости и статусе бронирования. Также присутствуют даты, поэтому данные можно анализировать во временном разрезе.

### Условный заказчик
Условным заказчиком анализа может выступать руководство гостиничной сети или команда по управлению бронированиями и доходностью.

### Возможные задачи интеллектуального анализа данных
1. Анализировать факторы, связанные с отменой бронирований.
2. Выполнять сегментацию бронирований по типу отеля, каналам продаж, типам клиентов и другим характеристикам.
3. Выявлять аномальные или подозрительные записи, например необычные значения стоимости или количества гостей.
4. Проводить описательную аналитику по сезонности и структуре бронирований.

## 2. Паспорт датасета

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

df = pd.read_csv("../data/hotel_bookings.csv")

print("Размер датасета:", df.shape)
display(df.head())


In [ ]:
# Типы и краткий паспорт признаков
passport = pd.DataFrame({
    "Признак": df.columns,
    "Тип pandas": [df[c].dtype for c in df.columns],
    "Количество уникальных": [df[c].nunique(dropna=True) for c in df.columns],
    "Пропуски": [df[c].isna().sum() for c in df.columns]
})
display(passport)


In [ ]:
# Приведение даты статуса бронирования к datetime
df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"], errors="coerce"
)

# Создание единой даты заезда из года, месяца и дня
months = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
df["arrival_month_num"] = df["arrival_date_month"].map(months)
df["arrival_date"] = pd.to_datetime(
    dict(
        year=df["arrival_date_year"],
        month=df["arrival_month_num"],
        day=df["arrival_date_day_of_month"]
    ),
    errors="coerce"
)

display(df[["arrival_date", "reservation_status_date"]].head())


## 3. Аудит качества данных
### 3.1. Пропуски

In [ ]:
missing = pd.DataFrame({
    "Количество пропусков": df.isna().sum(),
    "Доля пропусков, %": (df.isna().mean() * 100).round(2)
}).sort_values("Количество пропусков", ascending=False)

display(missing[missing["Количество пропусков"] > 0])


**Вывод.** Наибольшая проблема — столбец `company`: в нём отсутствуют значения примерно в 94,3% строк. Также заметная доля пропусков есть в `agent` (около 13,7%). В `country` пропуски составляют около 0,4%, а в `children` — только 4 записи. При дальнейшем анализе пропуски нужно обрабатывать с учётом смысла признака, а не автоматически заменять одним значением.

### 3.2. Дубликаты

In [ ]:
duplicate_count = df.duplicated().sum()
print("Количество полностью дублирующихся строк:", duplicate_count)
print("Доля дубликатов, %:", round(duplicate_count / len(df) * 100, 2))


**Вывод.** В датасете обнаружено 31 994 полностью дублирующиеся строки. Это существенно больше нуля, поэтому перед построением моделей необходимо определить, являются ли такие записи действительно повторными наблюдениями или отражают разные бронирования с одинаковым набором признаков.

### 3.3. Числовые и категориальные признаки

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
categorical_columns = df.select_dtypes(include="object").columns

print("Числовые признаки:")
display(df[numeric_columns].describe().T)

print("Категориальные признаки:")
cat_info = pd.DataFrame({
    "Уникальных значений": [df[c].nunique(dropna=True) for c in categorical_columns],
    "Пропусков": [df[c].isna().sum() for c in categorical_columns]
}, index=categorical_columns)
display(cat_info)


In [ ]:
# Проверка подозрительных значений
checks = pd.Series({
    "Отрицательный lead_time": (df["lead_time"] < 0).sum(),
    "Отрицательный ADR": (df["adr"] < 0).sum(),
    "Нулевая группа гостей": (
        (df["adults"].fillna(0) + df["children"].fillna(0) + df["babies"].fillna(0)) == 0
    ).sum(),
    "Нулевое количество взрослых": (df["adults"] == 0).sum(),
    "Необычные годы заезда": (~df["arrival_date_year"].isin([2015, 2016, 2017])).sum(),
    "Undefined в meal": (df["meal"] == "Undefined").sum(),
    "Undefined в market_segment": (df["market_segment"] == "Undefined").sum(),
    "Undefined в distribution_channel": (df["distribution_channel"] == "Undefined").sum()
})
display(checks.to_frame("Количество"))


**Вывод.** Обнаружена как минимум одна явно подозрительная запись с отрицательным `adr` (-6,38). Кроме того, 180 строк имеют нулевое суммарное количество гостей, а 403 строки — ноль взрослых. Такие записи требуют отдельной проверки перед дальнейшим анализом. Категории `Undefined` встречаются в нескольких категориальных признаках и также требуют документированного решения по обработке.

### 3.4. Выбросы методом IQR

In [ ]:
def iqr_outliers(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return q1, q3, iqr, lower, upper, mask

q1, q3, iqr, lower, upper, outlier_mask = iqr_outliers(df, "adr")
print(f"Q1 = {q1:.2f}")
print(f"Q3 = {q3:.2f}")
print(f"IQR = {iqr:.2f}")
print(f"Нижняя граница = {lower:.2f}")
print(f"Верхняя граница = {upper:.2f}")
print("Количество выбросов ADR:", outlier_mask.sum())

display(df.loc[outlier_mask, ["hotel", "adr", "adults", "children", "babies", "is_canceled"]].head(20))


In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot(df["adr"].dropna(), vert=False)
plt.xlabel("ADR")
plt.title("Выбросы в признаке ADR")
plt.show()


**Вывод.** По правилу IQR для `adr` получено 3 793 наблюдения вне интервала [−15,77; 211,07]. Это статистические выбросы, а не автоматически ошибки: часть высоких значений может отражать реальные дорогие бронирования. Отрицательное значение `adr` отдельно выглядит подозрительно и требует проверки.

## 4. Разведочный анализ

In [ ]:
# 1. Распределение ключевого числового признака
plt.figure(figsize=(10, 5))
plt.hist(df["lead_time"], bins=40)
plt.xlabel("Lead time, дней")
plt.ylabel("Количество бронирований")
plt.title("Распределение времени между бронированием и заездом")
plt.show()


**Комментарий.** Большая часть бронирований имеет относительно небольшое или среднее время предварительного бронирования, однако распределение имеет длинный правый хвост. Это показывает, что существуют бронирования, сделанные значительно заранее. Возникает гипотеза, что длительный `lead_time` может быть связан с вероятностью отмены.

In [ ]:
# 2. Распределение категориального признака
hotel_counts = df["hotel"].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(hotel_counts.index, hotel_counts.values)
plt.xlabel("Тип отеля")
plt.ylabel("Количество бронирований")
plt.title("Количество бронирований по типу отеля")
plt.xticks(rotation=15)
plt.show()


**Комментарий.** В датасете значительно больше бронирований для `City Hotel`, чем для `Resort Hotel`. Это означает, что общие показатели в первую очередь отражают структуру городского отеля. Для корректных сравнений полезно анализировать два типа отелей отдельно.

In [ ]:
# 3. Зависимость между признаками: отмена по типу отеля
cancel_by_hotel = df.groupby("hotel")["is_canceled"].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(cancel_by_hotel.index, cancel_by_hotel.values)
plt.xlabel("Тип отеля")
plt.ylabel("Доля отмен")
plt.title("Доля отмен бронирований по типу отеля")
plt.ylim(0, max(cancel_by_hotel.values) * 1.2)
plt.show()

display((cancel_by_hotel * 100).round(2).to_frame("Доля отмен, %"))


**Комментарий.** Доля отмен различается между двумя типами отелей: для `City Hotel` она составляет около 41,7%, для `Resort Hotel` — около 27,8%. Это наблюдение не доказывает причинную связь, но показывает, что тип отеля является полезным признаком для дальнейшего исследования отмен.

In [ ]:
# Дополнительная визуализация: сезонность
month_order = list(months.keys())
monthly = df["arrival_date_month"].value_counts().reindex(month_order)

plt.figure(figsize=(12, 5))
plt.bar(monthly.index, monthly.values)
plt.xlabel("Месяц заезда")
plt.ylabel("Количество бронирований")
plt.title("Распределение бронирований по месяцам заезда")
plt.xticks(rotation=45)
plt.show()


**Комментарий.** Количество бронирований заметно меняется по месяцам. Наибольшее число наблюдений приходится на август и июль, а наименьшее — на январь. Это указывает на выраженную сезонность и позволяет в дальнейшем исследовать связь сезона с типом отеля, стоимостью и отменами.

## Итоговые выводы

В результате первичного исследования установлено, что датасет содержит 119 390 строк и 32 исходных признака, включая числовые, категориальные и временные признаки. Основные проблемы качества — большое количество пропусков в `company`, заметные пропуски в `agent`, наличие пропусков в `country` и `children`, а также 31 994 полных дубликата.

Отдельной проверки требуют подозрительные значения: отрицательный `adr`, записи с нулевым количеством гостей и категории `Undefined`. Метод IQR выявил 3 793 статистических выброса в `adr`. На этапе дальнейшей обработки следует определить правила удаления или корректировки дубликатов и аномалий, отдельно решить вопрос с пропусками и проверить влияние очистки на результаты анализа.